# Creating AI without such libraries as Torch or TensorFlow

In [1]:
import json

save_path = "hackernews_texts.json"
with open(save_path, "r") as f:
    hackernews_texts = json.load(f)

In [2]:
import re

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9 .,!?]", "", text)  # Retain only valid characters
    return text

cleaned_texts = [clean_text(text) for text in hackernews_texts]
concatenated_text = "\n".join(cleaned_texts)

char_list = list()
for letter in concatenated_text:
    char_list.append(letter)

char_list[:10]
vocabulary = sorted(list(set(char_list)))

In [3]:
import numpy as np

stoi = {ch: number for number, ch in enumerate(vocabulary)}
encoded_text = [stoi[ch] for ch in concatenated_text]

tensored_data = np.array(encoded_text)
print(f"Encoded text length: {len(tensored_data)}")

n = int(0.9 * len(tensored_data))
train_data = tensored_data[:n]
val_data = tensored_data[n:]

print(f"Vocabulary length: {len(vocabulary)}")

Encoded text length: 61695
Vocabulary length: 68


In [4]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else val_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

In [5]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 128
vocabulary_size = len(vocabulary)
block_size = 64
batch_size = 32

sgd = Adam(lr=3e-4)
model = MiniGPT(vocab_size=vocabulary_size, d_model=128, block_size=64, n_layers=4, gradient=sgd)
xb, yb = get_batch("train", block_size, batch_size)

for step in range(5000):
    xb, yb = get_batch("train", block_size=64, batch_size=32)

    logits, loss = model.forward(xb, yb)
    
    model.backward()

    if step % 50 == 0:
        print("step:", step, "loss:", loss)

step: 0 loss: 4.187684587396321


KeyboardInterrupt: 

In [ ]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(len(vocabulary), p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [ ]:
itos = {i: ch for ch, i in stoi.items()}

prompt = "Hello"
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.int64)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 100)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

Hellog, SwiftUb app belops it befor the rest of the codebase for project is an butten type All, perhoning
